## Import libraries

In [1]:
import numpy as np
import os
import struct
from array import array
from prettytable import PrettyTable
from scipy.optimize import linear_sum_assignment

## data loader

In [2]:
class MNIST_loader:
    def __init__(self, MNIST_root):
        self.train_imgs_path = os.path.join(MNIST_root, 'train-images.idx3-ubyte_')
        self.test_imgs_path = os.path.join(MNIST_root, 't10k-images.idx3-ubyte')
        self.train_labels_path = os.path.join(MNIST_root, 'train-labels.idx1-ubyte_')
        self.test_labels_path = os.path.join(MNIST_root, 't10k-labels.idx1-ubyte')

    def read_data(self, mode = "train"):
        if mode == "train":
            imgs_path = self.train_imgs_path
            labels_path = self.train_labels_path
        elif mode == "test":
            imgs_path = self.test_imgs_path
            labels_path = self.test_labels_path
        
        imgs, labels = [], []
        with open(imgs_path, 'rb') as file:
            #read 16 bytes from file and read as binary in big-endian format 
            magic, size, rows, cols = struct.unpack(">IIII", file.read(16))
            if magic != 2051:
                raise ValueError('Magic number mismatch, expected 2051, got {}'.format(magic))
            print(f'Imgs | size = {size}, rows & cols = {rows} * {cols}')
            image_data = array("B", file.read())
        
        for i in range(size):
            #get one img every step
            img = np.array(image_data[i * rows * cols : (i + 1) * rows * cols])
            img = img.reshape(rows, cols)
            imgs.append(img)

        imgs = np.array(imgs)
        with open(labels_path, 'rb') as file:
            magic, size = struct.unpack(">II", file.read(8))
            if magic != 2049:
                raise ValueError('Magic number mismatch, expected 2049, got {}'.format(magic))
            print(f'Labels | size = {size}')
            labels = array("B", file.read())
        
        labels = np.array(labels)

        return imgs, labels

In [3]:
data_loader = MNIST_loader(".")
train_x, train_y = data_loader.read_data(mode = "train")

Imgs | size = 60000, rows & cols = 28 * 28
Labels | size = 60000


## EM Algorithm

In [4]:
class EM():
    def __init__(self, x, y):
        flatten_x = x.reshape(60000, 784)
        binary_flatten_x = (flatten_x > 127).astype(int)
        self.x, self.y = binary_flatten_x, y
        self.class_lambda = np.ones(10) / 10.
        self.class_pixel_probability = np.random.uniform(0.1, 0.9, (10, 784))
        self.w = np.zeros((60000, 10))
        self.count = 0
    def E_step(self):
        w = np.zeros((60000, 10))  # weight with class i (1 - 10) for each input img
        log_class_pixel_prob = np.log(self.class_pixel_probability + 1e-10)
        log_class_pixel_prob_neg = np.log(1 - self.class_pixel_probability + 1e-10)
        for c in range(10):
            log_prob = self.x * log_class_pixel_prob[c] + (1 - self.x) * log_class_pixel_prob_neg[c]
            w[:, c] = self.class_lambda[c] * np.exp(np.sum(log_prob, axis=1))
        w = w / (np.sum(w, axis=1, keepdims=True))
        return w
    def M_step(self, w):
        for c in range(10):
            self.class_lambda[c] = np.sum(w[:, c]) / 60000
            if np.sum(w[:, c]) > 0:
                self.class_pixel_probability[c, :] = np.sum(w[:, c][:, np.newaxis] * self.x, axis=0) / (np.sum(w[:, c]))
    def get_imaginations(self):
        for c in range(10):
            print('\n')
            print(f'class {c}:')
            pixels = self.class_pixel_probability[c].reshape(28, 28)
            table = PrettyTable()
            table.border = False
            table.header = False
            for row in pixels:
                table.add_row([("\033[91m1\033[0m" if pixel > 0.5 else "0") for pixel in row])
            table.align = "c"
            table.padding_width = 1
            print(table)
    def get_gt_probability(self):
        gt_p = np.zeros((10, 784))
        unique_labels, class_counts = np.unique(self.y, return_counts=True)
        for label in unique_labels:
            indices = np.where(self.y == label)[0]
            gt_p[label] = np.sum(self.x[indices], axis=0)
        for label in unique_labels:
            gt_p[label] /= class_counts[label]
        return gt_p
    def assign_labels_to_classes(self):
        gt_probability = self.get_gt_probability()
        distance = np.zeros((10, 10))
        for i in range(10):
            for j in range(10):
                distance[i, j] = np.linalg.norm(self.class_pixel_probability[i] - gt_probability[j])
        
        _, matching = linear_sum_assignment(distance)
        return matching
    def fit(self):
        difference = float('inf')
        while difference > 5:
            prob_old = self.class_pixel_probability.copy()
            self.w = self.E_step()
            self.M_step(self.w)
            self.get_imaginations()
            self.count += 1
            difference = np.sum(np.abs(prob_old - self.class_pixel_probability))
            print(f'No. of Iteration: {self.count}, Difference: {difference}')
            print('--------------------------------------------------------------------------------------')
            print('--------------------------------------------------------------------------------------')
    def get_confusion_matrix(self, digit, class_num, y_pred):
        y_pred = y_pred.flatten()
        y_gt = self.y.flatten()
        actual_class = (y_gt == digit).astype(int)
        predicted_class = (y_pred == class_num).astype(int)
        tp = np.sum((actual_class == 1) & (predicted_class == 1))
        tn = np.sum((actual_class == 0) & (predicted_class == 0))
        fp = np.sum((actual_class == 0) & (predicted_class == 1))
        fn = np.sum((actual_class == 1) & (predicted_class == 0))
        return tp, tn, fp, fn
    def get_final_output(self):
        matching = self.assign_labels_to_classes()
        class_order = []
        for digit in range(10):
            class_order.append(np.where(matching == digit)[0])
        for digit in range(10):
            print(f'labeled class {digit}:')
            pixels = self.class_pixel_probability[class_order[digit]].reshape(28, 28)
            table = PrettyTable()
            table.border = False
            table.header = False
            for row in pixels:
                table.add_row([("\033[91m1\033[0m" if pixel > 0.5 else "0") for pixel in row])
            table.align = "c"
            table.padding_width = 1
            print(table)
        y_pred = np.argmax(self.w, axis = 1)
        error = 0
        for digit in range(10):
            tp, tn ,fp, fn= self.get_confusion_matrix(digit, class_order[digit], y_pred)
            print(f'Confusion Matrix: \n             Predict number {digit} Predict not number {digit}')
            print(f'Is number {digit}           {tp}               {fn}')
            print(f'Isn\'t number {digit}        {fp}               {tn}\n')
            print(f'Sensitivity (Successfully predict number {digit}): {tp / (tp + fp): .5f}')
            print(f'Specificity (Successfully predict not number {digit}): {tn / (tn + fn): .5f}')
            print('\n----------------------------------------------------------------')
            error += (fp / (tp + fp) + fn / (tn + fn))
        error /= 10
        print(f'Total iteration to converge: {self.count}')
        print(f'Total error rate: {error} ')

## Experiment

In [5]:
em = EM(train_x, train_y)
em.fit()



class 0:
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  1  0  0  0  0  0  0  0  0  0  0  0 
 0  0  0  0  0  0  0  0  0  0  0  0  1  1  1  1  1  1  0  0  0  0  0  0  0  0  0  0 
 0  0  0  0  0  0  0  0  0  0  0  1  1  1  1  1  1  1  1  0  0  0  0  0  0  0  0  0 
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  1  0  0  0  0  0  0  0  0  0  0 
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 

In [6]:
em.get_final_output()

labeled class 0:
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  1  1  1  0  0  0  0  0  0  0  0  0  0 
 0  0  0  0  0  0  0  0  0  0  0  0  1  1  1  1  1  1  1  1  0  0  0  0  0  0  0  0 
 0  0  0  0  0  0  0  0  0  0  0  1  1  1  1  1  1  1  1  1  1  0  0  0  0  0  0  0 
 0  0  0  0  0  0  0  0  0  0  1  1  1  1  1  1  1  1  1  1  1  1  0  0  0  0  0  0 
 0  0  0  0  0  0  0  0  0  1  1  1  1  1  0  0  0  0  0  1  1  1  0  0  0  0  0  0 
 0  0  0  0  0  0  0  0  0  1  1  1  1  0  0  0  0  0  0  0  1  1  1  0  0  0  0  0 
 0  0  0  0  0  0  0  0  1  1  1  1  0  0  0  0 